In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks'
METADATA_CSV = f'{DATA_DIR}/HAM10000_metadata.csv'
IMG_PART1 = f'{DATA_DIR}/HAM10000_images_part_1'
IMG_PART2 = f'{DATA_DIR}/HAM10000_images_part_2'

## 1.5) Verify dataset

In [ ]:
import glob
import os

missing = []
for path in [METADATA_CSV, IMG_PART1, IMG_PART2]:
    if not os.path.exists(path):
        missing.append(path)

if missing:
    print('Missing paths:', missing)
    print('Check Drive mount and folder name in DATA_DIR.')
    if os.path.exists('/content/drive/MyDrive'):
        print('Drive root:', os.listdir('/content/drive/MyDrive')[:20])
    raise FileNotFoundError(f"Missing paths: {missing}")

count1 = len(glob.glob(os.path.join(IMG_PART1, '*.jpg')))
count2 = len(glob.glob(os.path.join(IMG_PART2, '*.jpg')))
print('Images part 1:', count1)
print('Images part 2:', count2)
print('Total images:', count1 + count2)

## 2) Install dependencies

In [ ]:
!pip -q install pandas scikit-learn torch torchvision pillow

## 3) Train EfficientNet-B0

In [ ]:
import json
import os
import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

LABELS = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
EPOCHS = 8
BATCH_SIZE = 32
LR = 1e-4
IMAGE_SIZE = 224
NUM_WORKERS = 2
FREEZE_BACKBONE = False
OUTPUT_DIR = '/content/drive/MyDrive/dermai_model'

class HamDataset(Dataset):
    def __init__(self, items, image_size=224):
        self.items = items
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        image_path, label = self.items[index]
        image = Image.open(image_path).convert('RGB')
        return self.transform(image), label

def resolve_image_path(image_id):
    filename = f'{image_id}.jpg'
    part1 = os.path.join(IMG_PART1, filename)
    if os.path.exists(part1):
        return part1
    part2 = os.path.join(IMG_PART2, filename)
    if os.path.exists(part2):
        return part2
    raise FileNotFoundError(filename)

df = pd.read_csv(METADATA_CSV)
label_to_idx = {label: idx for idx, label in enumerate(LABELS)}
items = []
for _, row in df.iterrows():
    image_path = resolve_image_path(row['image_id'])
    items.append((image_path, label_to_idx[row['dx']]))

y = [label for _, label in items]
train_items, val_items = train_test_split(items, test_size=0.2, stratify=y, random_state=42)

train_loader = DataLoader(HamDataset(train_items, image_size=IMAGE_SIZE), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(HamDataset(val_items, image_size=IMAGE_SIZE), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
if FREEZE_BACKBONE:
    for param in model.features.parameters():
        param.requires_grad = False
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(LABELS))
model.to(device)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
criterion = nn.CrossEntropyLoss()

os.makedirs(OUTPUT_DIR, exist_ok=True)
best_model_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
labels_path = os.path.join(OUTPUT_DIR, 'labels.json')

best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    val_acc = correct / max(total, 1)
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
    print(f'Epoch {epoch + 1}/{EPOCHS} | loss: {running_loss / len(train_items):.4f} | val_acc: {val_acc:.4f}')

with open(labels_path, 'w') as handle:
    json.dump({'labels': LABELS}, handle, indent=2)

print('Best val acc:', best_acc)
print('Saved to:', best_model_path)

## 4) Download model files

In [ ]:
from google.colab import files
files.download(f'{OUTPUT_DIR}/best_model.pth')
files.download(f'{OUTPUT_DIR}/labels.json')